# Graph Convolutional Network (GCN) on Cora

This notebook tackles semi-supervised node classification on the Cora citation network, where each node is a paper represented by a bag-of-words feature vector and the goal is to predict its topic class using only a handful of labeled nodes plus the citation graph structure; the approach follows Kipf & Welling's Graph Convolutional Network, stacking two `GCNConv` layers (with dropout for regularization) that propagate and transform features along graph edges so that a node's prediction is informed by its neighborhood, trained end-to-end with the Adam optimizer against a masked cross-entropy loss for 200 epochs and evaluated by classification accuracy on held-out test nodes. The single code cell below installs **K3-Node**, loads Cora through `k3_node.datasets.Planetoid`, defines the `K3GCN` model (two `GCNConv` layers plus dropout) on **Keras 3**, compiles and trains it through a small data generator, and finally reports test accuracy — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q git+http://github.com/anas-rz/k3-node/@main

import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid
from k3_node import transforms as k3_transforms

title = "Graph Convolutional Network (GCN) on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=k3_transforms.NormalizeFeatures())
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. GCN Model Definition
class K3GCN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.GCNConv(hidden_channels, out_channels)
        self.dropout = layers.Dropout(0.5)

    def call(self, inputs, edge_index=None, training=False):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = self.dropout(x, training=training)
        x = ops.relu(self.conv1(x, edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, edge_index)

k3_model = K3GCN(num_features, 16, num_classes)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01, weight_decay=5e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 4. Generator & Training
def graph_data_generator():
    x = ops.convert_to_tensor(data.x, dtype="float32")
    edge_index = ops.convert_to_tensor(data.edge_index, dtype="int64")
    y = ops.convert_to_tensor(data.y, dtype="int64")
    mask = ops.cast(data.train_mask, "float32")
    while True:
        yield (x, edge_index), y, mask

print(f"Training K3-Node GCN on {backend} backend...")
history = k3_model.fit(
    graph_data_generator(),
    steps_per_epoch=1,
    epochs=20,
    verbose=1,
)

# 5. Evaluation
out = k3_model((data.x, data.edge_index))
pred = ops.argmax(out, axis=-1)
test_mask = data.test_mask
test_acc = ops.mean(ops.cast(ops.cast(pred[test_mask], "int64") == ops.cast(data.y[test_mask], "int64"), "float32"))
print(f"Test Accuracy: {float(test_acc):.4f}")

print("\n✓ K3-Node execution completed successfully!")
